# 03. Returns, delistings and market capitalisation

**What this notebook does.** For every universe member in every month it builds the monthly return with any delisting return applied under rule D4, and the market capitalisation (price times shares outstanding) at the start of the month. It counts what rule D4 and the rules on missing values decide, and judges gate G4 on the capitalisation. It then reads the daily stock file once, a year at a time, to judge gate G3 on the stocks that leave the universe and to measure how many member-days have a daily return.

**Why it comes here.** The benchmark of notebook 04 is a weighted sum of these returns with these capitalisations as weights. A delisting left out would drop the loss on the stocks that fail. A capitalisation read at the end of the month would weight each stock by a price not yet known when the month began.

**Terms used in this notebook.**

- *universe*: the stocks in CRSP's list of index members on the last trading day of the month before.
- *member-month*: one stock in the universe in one month.
- *return*: CRSP's monthly return with dividends (`ret`). The *price-only return* (`retx`) leaves dividends out.
- *delisting*: the end of a stock's listing on its exchange. CRSP records its date, a code for its reason (`dlstcd`) and, where it can, the *delisting return* (`dlret`), the return from the last price to the value the holder received.
- *rule D4*: the study's rule for a delisting, after Shumway (1997). CRSP's delisting return where it gives one. Where it is missing, minus 30 per cent for a code from 400 to 599, the codes for a delisting for poor performance, and zero for any other code. CRSP's code 100 marks a security still trading and is not a delisting. In a month with both a return and a delisting return, the two are compounded.
- *market capitalisation*: price times shares outstanding. The capitalisation at the start of month t is the absolute price times shares outstanding at the end of month t−1, in thousands of dollars. CRSP stores the price as negative when it is the midpoint of bid and ask on a day without a trade, hence the absolute value. A price of zero or a missing value gives a missing capitalisation.
- *usable window*: the months from the first in which the universe's count is within 10 names of CRSP's count, to December 2024, as notebook 02 found it.
- *gate*: a test of the data layer, fixed in advance in `src/slt/constants.py`. A gate that is not met stops the study.
- *verdict*: the result of a gate, met or not met, printed in the same words either way.
- *removal*: the end of a membership spell before 31 December 2024. A spell that ends on that date belongs to a current member.
- *trading day*: a day on which the daily stock file has rows.
- *member-day*: a trading day in a month in which the stock is in the universe.

**What I expect to see, and where it comes from.** From the manifest: the monthly stock file has 5,153,763 rows, the delisting file 38,843, and the daily file 37 yearly files, of which the 35 from 1990 to 2024 hold rows, from 2 January 1990 to 31 December 2024. The files for 2025 and 2026 are empty, because CRSP ends on 31 December 2024 on this subscription. From a structural count of 10 November 2026: 9,737 rows of the delisting file carry code 100, and no PERMNO has more than one row there.

- *Rule D4.* A stock usually leaves the index before it fails, so in the universe a delisting is expected to be a merger or an acquisition (codes 200 to 399) with CRSP's delisting return given, and the minus 30 per cent to be used rarely. Every count is printed by year and committed. None has a verdict.
- *Missing values.* A member with no return and no delisting in a month, and a member without a capitalisation, are expected close to zero. Each is counted by year.
- *G4.* CRSP reports price and shares for nearly every listed stock-month, so G4, a capitalisation in at least 90 per cent of a name's membership months for at least 90 per cent of names over the usable window, is expected met with room.
- *G3.* A stock removed from the index usually keeps trading on its exchange, so a price within 10 trading days of the removal is expected for nearly every removal from 2 January 1990, and G3, at least 90 per cent, is expected met with room. Removals before 1990 have no daily file. They are reported on the monthly file with no verdict.
- *Daily coverage.* Member-days with a return are expected close to all member-days in every year, reported with no verdict.

Running time: the monthly part takes about a minute. The daily pass reads the 35 yearly files from 1990 to 2024, 2.2 GB on disk, and prints each year as it finishes.

In [ ]:
import json
import pathlib
import sys
import time

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd

from slt import constants as C
from slt import extraction as X

root = X.data_root()             # stops with one sentence when the data root is not set
manifest = X.check_manifest(root)   # stops when the manifest on disk is not the one the constants fix
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}")

In [ ]:
from slt import layer as L
from slt import returns as R
from slt import sectors as S
from slt import universe as U
from slt import validation as V

## 1. The monthly panel

The panel holds one row per member-month: the month's return and price-only return with any delisting applied, the capitalisation from the month before, and how the month's delisting, if any, was treated. It is built in memory from the files on every run and never written. Its number of rows has to equal the number of member-months, because each member-month takes at most one row of each file.

In [ ]:
window = X.read_result("usable_window.json")
t0 = time.time()
universe = L.load_universe(root)
data = L.load_panel(root, universe)
panel, monthly = data["panel"], data["monthly"]
repeated = int(monthly.assign(m=pd.PeriodIndex(monthly["date"], freq="M")).duplicated(["permno", "m"]).sum())
print(f"member-months {len(universe['ulong']):,} | panel rows {len(panel):,} | equal: {len(panel) == len(universe['ulong'])}")
print("PERMNO-months with more than one row in the monthly file:", repeated)
print("date values present that did not parse:", {**universe["unparsed"], **data["unparsed"]} or "none")
print(f"usable window from {window['first_month']} | {time.time() - t0:.0f} s")

## 2. Delistings under rule D4

Every delisting month of a member is counted by how its return was set: CRSP's delisting return, minus 30 per cent, or zero. The months with neither a return nor a delisting, and the months without a capitalisation, are counted beside them. Each such member is left out of its month's benchmark return in notebook 04, and the other weights are rescaled.

In [ ]:
d4 = R.delisting_by_year(panel)
print(d4.drop(columns="year").sum().to_string())
families = R.delisting_code_families(panel)
print(families.to_string())
print(d4[d4["delisting_months"] > 0].tail(10).to_string(index=False))

## 3. Market capitalisation and gate G4

For each name, the share of its membership months inside the usable window that have a capitalisation. G4 asks that at least 90 per cent of names have one in at least 90 per cent of their months.

In [ ]:
g4 = R.cap_coverage(panel, window["first_month"])
print(json.dumps(g4, indent=1))
print(V.verdict_line("G4", g4["g4_met"], f"{g4['names_meeting_month_share']:,} of {g4['names']:,} names ({g4['share_of_names']:.2%}) have a "
      f"capitalisation in at least {C.G4_MONTH_SHARE_MIN:.0%} of their membership months, against a level of {C.G4_NAME_SHARE_MIN:.0%} of names."))

## 4. Removals and the daily file

One pass over the daily file, a year at a time, keeps three things from each year: its trading days, the member-days with a return, and the priced rows of each removed stock within 40 calendar days of its removal, which holds every window of 10 trading days. Nothing else of the file stays in memory, and nothing of it is written.

In [ ]:
removed = R.removals(universe["spells"])
in_span = (removed["ending"] >= pd.Timestamp(C.G3_WINDOW[0])) & (removed["ending"] <= pd.Timestamp(C.G3_WINDOW[1]))
print("removals", len(removed), "| from", C.G3_WINDOW[0], "to", C.G3_WINDOW[1], int(in_span.sum()), "| before", int((~in_span).sum()))
t0 = time.time()
daily = L.daily_pass(root, universe["ulong"], removed, progress=True)
calendar = pd.DatetimeIndex(daily["calendar"])
print(f"read in {time.time() - t0:.0f} s | trading days {len(calendar):,}, {calendar[0].date()} to {calendar[-1].date()}"
      f" | priced rows kept near a removal {len(daily['removed_rows']):,}")
print("date values present that did not parse:", daily["unparsed"] or "none")

## 5. Gate G3

For each removal from 2 January 1990, the window runs from 10 trading days before the last trading day on or before the removal date to 10 trading days after it. G3 asks that at least 90 per cent of removals have a price inside their window. Removals before the daily file are reported on the monthly file, as the share with a price at the end of the removal month or of the month before, with no verdict.

In [ ]:
windows = R.removal_windows(removed, daily["calendar"])
found = R.priced_near(windows, daily["removed_rows"])
g3 = R.g3_verdict(windows, found)
before = R.removals_before_daily_file(removed, monthly)
print(V.verdict_line("G3", g3["g3_met"], f"{g3['removals_with_price']:,} of {g3['removals']:,} removals from {C.G3_WINDOW[0]} to {C.G3_WINDOW[1]} "
      f"({g3['share']:.2%}) have a price within {C.G3_TRADING_DAYS} trading days, against a level of {C.G3_SHARE_MIN:.0%}."))
print("Before the daily file, with no verdict:", json.dumps(before))

## 6. Daily returns of the members

By year: the member-days expected, which is each month's number of members times its trading days, against the member-days with a return. A member that joins or leaves the exchange inside a month has fewer days than the month, so the share sits a little below one even when nothing is missing.

In [ ]:
days = R.trading_days_by_month(daily["calendar"])
coverage = R.daily_coverage_by_year(daily["present"], days, universe["ulong"])
print(coverage.to_string(index=False))

## 7. What is committed

Five files go into `results/20260928/`, each an aggregate with no identifier of a security: rule D4's counts by year, G4's figures and verdict, G3's figures and verdict with the removals before 1990, the daily coverage by year, and a summary of the counts printed in this notebook.

In [ ]:
summary = {
    "written_by": "notebooks/03_returns.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
    "member_months": int(len(universe["ulong"])), "panel_rows": int(len(panel)),
    "permno_months_with_more_than_one_monthly_row": repeated,
    "delisting_months_by_code_family": {str(k): int(v) for k, v in families.items()},
    "trading_days": int(len(calendar)), "first_trading_day": str(calendar[0].date()), "last_trading_day": str(calendar[-1].date()),
    "date_values_not_parsed": {**universe["unparsed"], **data["unparsed"], **daily["unparsed"]},
}
written = [X.write_results("delistings_by_year.csv", frame=d4),
           X.write_results("capitalisation_g4.json", record=g4),
           X.write_results("removals_g3.json", record={"in_daily_file": g3, "before_daily_file": before}),
           X.write_results("daily_coverage_by_year.csv", frame=coverage),
           X.write_results("returns_summary.json", record=summary)]
for path in written:
    print("written:", path.relative_to(ROOT))

## What was learned, and what could be wrong

What the notebook establishes: how each delisting month of a member was treated and how many months rule D4's minus 30 per cent decides, how many member-months have no return or no capitalisation, the verdicts of G3 and G4, and the share of member-days with a daily return.

What could still be wrong. Rule D4's minus 30 per cent is an average loss estimated by Shumway (1997) on stocks delisted for performance, and it is applied to every such month with no delisting return, whatever the stock's own loss was. A price within 10 trading days of a removal shows that the removed stock can be valued near the day it leaves. It does not show that the price is the one at which it left. And a capitalisation from shares outstanding counts every share, where the index itself has counted only the shares available to the public since 2005, so the weights here are CRSP's and differ from the index's own.

Shumway, T. (1997). The delisting bias in CRSP data. *Journal of Finance* 52(1), 327 to 340.